# EDA: 20 Newsgroups, RCV1/RCV2, LectureBank

## Setup

In [ ]:
# Install packages
!pip install -q datasets nltk


In [ ]:
# Imports
import os
import re
import glob
import tarfile
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy.sparse import lil_matrix

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize


In [ ]:
# Helper functions used across all datasets

def basic_stats(docs, name):
    # Returns dict of basic length statistics
    n = len(docs)
    char_lens = [len(d) for d in docs]
    word_lens = [len(d.split()) for d in docs]
    return {
        'dataset': name,
        'n_docs': n,
        'chars_mean': np.mean(char_lens),
        'chars_median': np.median(char_lens),
        'chars_min': np.min(char_lens),
        'chars_max': np.max(char_lens),
        'words_mean': np.mean(word_lens),
        'words_median': np.median(word_lens),
        'words_min': np.min(word_lens),
        'words_max': np.max(word_lens),
    }

def sentence_counts(docs, sample_size=None):
    # Count sentences per doc; sample if dataset is large
    if sample_size and len(docs) > sample_size:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(docs), size=sample_size, replace=False)
        sampled = [docs[i] for i in idx]
    else:
        sampled = docs
    return [len(sent_tokenize(d)) for d in sampled]

def length_distribution_plot(word_lens, title):
    # Histogram of word lengths clipped at 99th percentile
    cap = int(np.percentile(word_lens, 99))
    plt.figure(figsize=(7, 3))
    plt.hist(np.clip(word_lens, 0, cap), bins=50)
    plt.xlabel('Words per document (clipped at 99th pctl)')
    plt.ylabel('Frequency')
    plt.title(title)
    plt.tight_layout()
    plt.show()

def vocab_stats(docs, sample_size=20000):
    # Vocabulary statistics on a sample
    if len(docs) > sample_size:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(docs), size=sample_size, replace=False)
        sampled = [docs[i] for i in idx]
    else:
        sampled = docs
    counter = Counter()
    total_tokens = 0
    for d in sampled:
        toks = re.findall(r"[A-Za-z']+", d.lower())
        counter.update(toks)
        total_tokens += len(toks)
    vocab_size = len(counter)
    hapax = sum(1 for _, c in counter.items() if c == 1)
    return {
        'sample_docs': len(sampled),
        'total_tokens': total_tokens,
        'vocab_size': vocab_size,
        'hapax_count': hapax,
        'hapax_ratio': hapax / vocab_size if vocab_size else 0,
        'top_20': counter.most_common(20),
    }

def short_noisy_proportion(docs, min_words=20, min_sents=2):
    # Proportion of docs too short for sentence-level modelling
    too_short_words = sum(1 for d in docs if len(d.split()) < min_words)
    if len(docs) <= 5000:
        sc = [len(sent_tokenize(d)) for d in docs]
        denom = len(docs)
    else:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(docs), size=5000, replace=False)
        sc = [len(sent_tokenize(docs[i])) for i in idx]
        denom = 5000
    too_few_sents = sum(1 for c in sc if c < min_sents)
    return {
        'n_total': len(docs),
        'too_short_words_count': too_short_words,
        'too_short_words_pct': 100 * too_short_words / len(docs),
        'too_few_sents_pct (sampled)': 100 * too_few_sents / denom,
    }

def show_examples(docs, labels=None):
    # Show one short, one medium, one long doc by word count
    word_lens = np.array([len(d.split()) for d in docs])
    short_idx = int(np.argmin(word_lens))
    long_idx = int(np.argmax(word_lens))
    median_val = np.median(word_lens)
    med_idx = int(np.argmin(np.abs(word_lens - median_val)))
    for tag, i in [('SHORT', short_idx), ('MEDIUM', med_idx), ('LONG', long_idx)]:
        print(f"--- {tag} (words={word_lens[i]}) ---")
        if labels is not None:
            print(f"label: {labels[i]}")
        print(docs[i][:600])
        print()


## 1. 20 Newsgroups

In [ ]:
# Load 20 Newsgroups via sklearn
from sklearn.datasets import fetch_20newsgroups

# remove headers/footers/quotes for cleaner body text
ng_clean = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

ng_docs = ng_clean.data
ng_labels = [ng_clean.target_names[t] for t in ng_clean.target]
print('docs:', len(ng_docs), '| classes:', len(ng_clean.target_names))


In [ ]:
# Document count + basic length stats
ng_stats = basic_stats(ng_docs, '20 Newsgroups')
pd.Series(ng_stats)


In [ ]:
# Class distribution
ng_class_counts = pd.Series(ng_labels).value_counts()
print(ng_class_counts)
ng_class_counts.plot(kind='bar', figsize=(10, 3), title='20NG class distribution')
plt.tight_layout(); plt.show()


In [ ]:
# Document length distribution (in words)
ng_word_lens = [len(d.split()) for d in ng_docs]
length_distribution_plot(ng_word_lens, '20NG document length (words)')


In [ ]:
# Sentence count distribution
ng_sent_counts = sentence_counts(ng_docs)
plt.figure(figsize=(7, 3))
plt.hist(np.clip(ng_sent_counts, 0, np.percentile(ng_sent_counts, 99)), bins=50)
plt.title('20NG sentences per document'); plt.xlabel('sentences'); plt.ylabel('freq')
plt.tight_layout(); plt.show()
print('mean sents:', np.mean(ng_sent_counts), 'median:', np.median(ng_sent_counts))


In [ ]:
# Vocabulary statistics
ng_vocab = vocab_stats(ng_docs)
for k, v in ng_vocab.items():
    if k != 'top_20':
        print(k, ':', v)
print('top 20 tokens:', ng_vocab['top_20'])


In [ ]:
# Examples short/medium/long
show_examples(ng_docs, ng_labels)


In [ ]:
# Proportion too short / too noisy for sentence-level modelling
print(short_noisy_proportion(ng_docs))


## 2. RCV1/RCV2 (UCI)

In [ ]:
# Download RCV1/RCV2 dataset directly from UCI
url = 'https://archive.ics.uci.edu/static/public/259/reuters+rcv1+rcv2+multilingual+multiview+text+categorization+test+collection.zip'
print('downloading...')
r = requests.get(url, timeout=120)
print('status:', r.status_code, 'bytes:', len(r.content))

# Save and extract zip
with open('rcv1rcv2.zip', 'wb') as f:
    f.write(r.content)

with zipfile.ZipFile('rcv1rcv2.zip', 'r') as z:
    z.extractall('rcv1rcv2_data')

# Check what was extracted
!find rcv1rcv2_data/ -type f | head -20


In [ ]:
# The archive contains a .tar.bz2 inside — extract that too
tar_files = glob.glob('rcv1rcv2_data/**/*.tar.bz2', recursive=True)
if tar_files:
    print('extracting:', tar_files[0])
    with tarfile.open(tar_files[0], 'r:bz2') as t:
        t.extractall('rcv1rcv2_data')
else:
    print('no tar.bz2 found, checking structure:')
    !find rcv1rcv2_data/ -type d

# List extracted structure
!find rcv1rcv2_data/ -type f | head -30


In [ ]:
# Load English original documents (Index_EN-EN)
# Format per line: <label> <feature>:<value> <feature>:<value> ...
en_file = glob.glob('rcv1rcv2_data/**/Index_EN-EN', recursive=True)
if not en_file:
    en_file = glob.glob('rcv1rcv2_data/**/EN/Index_EN-EN', recursive=True)
print('found:', en_file)

# Parse SVMlight-style format
rcv_labels = []
rcv_rows = []
with open(en_file[0], 'r') as f:
    for line in f:
        parts = line.strip().split()
        if not parts:
            continue
        rcv_labels.append(parts[0])
        feat_dict = {}
        for p in parts[1:]:
            if ':' in p:
                idx, val = p.split(':')
                feat_dict[int(idx)] = float(val)
        rcv_rows.append(feat_dict)

print('English docs loaded:', len(rcv_labels))
print('label distribution:')
print(pd.Series(rcv_labels).value_counts())


In [ ]:
# Convert to sparse matrix for analysis
max_feat = max(max(r.keys()) for r in rcv_rows if r) + 1
X_rcv = lil_matrix((len(rcv_rows), max_feat))
for i, r in enumerate(rcv_rows):
    for idx, val in r.items():
        X_rcv[i, idx] = val
X_rcv = X_rcv.tocsr()
y_rcv = pd.Series(rcv_labels)

print('feature matrix shape:', X_rcv.shape)
print('classes:', y_rcv.nunique(), y_rcv.unique().tolist())


In [ ]:
# Document count
print('total documents (English):', X_rcv.shape[0])


In [ ]:
# Document length proxy: number of non-zero features per doc
nnz_per_doc = np.asarray((X_rcv != 0).sum(axis=1)).ravel()
print('mean non-zero features/doc:', nnz_per_doc.mean())
print('median:', np.median(nnz_per_doc))
print('min:', nnz_per_doc.min(), 'max:', nnz_per_doc.max())


In [ ]:
# Document length distribution (non-zero features as proxy)
plt.figure(figsize=(7, 3))
cap = int(np.percentile(nnz_per_doc, 99))
plt.hist(np.clip(nnz_per_doc, 0, cap), bins=50)
plt.title('RCV1/RCV2 (EN) non-zero features per doc (length proxy)')
plt.xlabel('non-zero features'); plt.ylabel('freq')
plt.tight_layout(); plt.show()


In [ ]:
# Class/topic distribution (6 categories)
class_counts = y_rcv.value_counts()
print('classes:', len(class_counts))
print(class_counts)
class_counts.plot(kind='bar', figsize=(7, 3), title='RCV1/RCV2 (EN) class distribution')
plt.tight_layout(); plt.show()


In [ ]:
# Vocabulary statistics from feature matrix
vocab_size = X_rcv.shape[1]
doc_freq = np.asarray((X_rcv != 0).sum(axis=0)).ravel()
hapax = (doc_freq == 1).sum()
common = (doc_freq > 0.5 * X_rcv.shape[0]).sum()
print('vocab size (features):', vocab_size)
print('hapax-like (in only 1 doc):', hapax)
print('tokens in >50%% of docs:', common)


In [ ]:
# Note on raw text and sentence-level analysis
# This UCI dataset provides feature vectors, not raw text.
# Sentence count distribution, document length in words, and
# short/medium/long examples cannot be computed from feature vectors.
print('Raw text not available in this UCI distribution.')
print('Sentence counts and text examples not applicable.')
print('Dataset contains 5 languages: EN, FR, GR, IT, SP (6 categories each).')
print('EDA above is for English (EN-EN) subset only.')


## 3. LectureBank

In [ ]:
# Load LectureBank from HuggingFace
from datasets import load_dataset

lb_hf = load_dataset("Nightbutterfly/LectureBank", split="train")
lb = lb_hf.to_pandas()
print('shape:', lb.shape)
print('columns:', lb.columns.tolist())
lb.head()


In [ ]:
# Load taxonomy from GitHub repo for topic name mapping
!rm -rf lecturebank_repo
!git clone --depth 1 https://github.com/Yale-LILY/LectureBank.git lecturebank_repo

tax = pd.read_csv('lecturebank_repo/taxonomy.csv')
print('taxonomy columns:', tax.columns.tolist())
print('taxonomy entries:', len(tax))
tax.head()


In [ ]:
# Document count
print('lecture entries:', len(lb))


In [ ]:
# Document text proxy from titles
# (actual slide content requires downloading linked PDFs/PPTs)
lb_docs = lb['Title'].astype(str).tolist()

lb_stats = basic_stats(lb_docs, 'LectureBank (titles)')
pd.Series(lb_stats)


In [ ]:
# Topic distribution
# Topic IDs in data are hierarchical codes (e.g. 133 = top-level 1, sub 3, sub 3)
# First digit maps to top-level taxonomy category
tax_top = dict(zip(tax.iloc[:, 0], tax.iloc[:, 1]))

lb['top_topic_id'] = lb['Topic'].astype(str).str[0].astype(int)
lb['top_topic_name'] = lb['top_topic_id'].map(tax_top)

# Full granular topic code count
topic_counts = lb['Topic'].value_counts()
print('unique topic codes in data:', len(topic_counts))
print()

# Top-level topic distribution
top_topic_counts = lb['top_topic_name'].value_counts()
print('top-level topic distribution:')
print(top_topic_counts)
top_topic_counts.plot(kind='bar', figsize=(10, 3), title='LectureBank top-level topic distribution')
plt.tight_layout(); plt.show()


In [ ]:
# Full granular topic code distribution (top 30)
topic_counts.head(30).plot(kind='bar', figsize=(12, 3), title='LectureBank top 30 topic codes')
plt.tight_layout(); plt.show()


In [ ]:
# Document length distribution (titles are short)
lb_word_lens = [len(d.split()) for d in lb_docs]
plt.figure(figsize=(7, 3))
plt.hist(lb_word_lens, bins=range(0, max(lb_word_lens)+2))
plt.title('LectureBank title length (words)'); plt.xlabel('words'); plt.ylabel('freq')
plt.tight_layout(); plt.show()
print('mean words:', np.mean(lb_word_lens), 'median:', np.median(lb_word_lens))


In [ ]:
# Sentence count distribution (titles are typically single fragments)
lb_sent_counts = [len(sent_tokenize(d)) for d in lb_docs]
print('sentence counts (titles):', Counter(lb_sent_counts))


In [ ]:
# Vocabulary statistics on titles
lb_vocab = vocab_stats(lb_docs)
for k, v in lb_vocab.items():
    if k != 'top_20':
        print(k, ':', v)
print('top 20 tokens:', lb_vocab['top_20'])


In [ ]:
# Examples short/medium/long
show_examples(lb_docs)


In [ ]:
# Proportion too short / too noisy for sentence-level modelling
print(short_noisy_proportion(lb_docs))
print()
print('Note: titles alone are too short for sentence-level modelling.')
print('Real sentence-level EDA needs the linked PDFs/PPTs downloaded and parsed.')


## Summary

In [ ]:
# Combined basic stats for datasets with raw text
summary = pd.DataFrame([ng_stats, lb_stats])
summary


In [ ]:
# RCV1/RCV2 summary (feature vectors only, no raw text)
rcv_summary = {
    'dataset': 'RCV1/RCV2 (UCI, English)',
    'n_docs': X_rcv.shape[0],
    'n_features': X_rcv.shape[1],
    'mean_nonzero_features_per_doc': float(nnz_per_doc.mean()),
    'n_classes': int(y_rcv.nunique()),
    'classes': y_rcv.unique().tolist(),
}
pd.Series(rcv_summary)
